# **VoxCity Demo**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kunifujiwara/VoxCity/blob/main/docs/examples/demo_quickstart.ipynb)

You can easily learn how the Python package "VoxCity" works by following this demo.

Please execute cells from Step 1 to Step 6 by clicking the run icons.

It is important to note that you need to choose either Option 1 or Option 2 in Step 2 and execute only one.

**No account is needed.** The default data sources (OpenStreetMap, Microsoft and Overture buildings, Meta/WRI canopy height, ESA WorldCover, USGS 3DEP, FABDEM and Copernicus terrain) are read directly from open repositories. Google Earth Engine (Step 1.2) is optional and only adds further sources.

# Step1. Prepare environment

In [ ]:
#@title 1.1. Install voxcity
!pip install voxcity

In [ ]:
#@title (Optional) 1.2. Authenticate Google Earth Engine - skip this cell unless you want Earth Engine sources
# !earthengine authenticate --auth_mode=notebook   # Google Colab: click the link, generate a token, paste it

# import ee                                        # local environment
# ee.Authenticate()
# ee.Initialize(project='your-project-id')

# Step2. Set target area by a rectangle

## 2.1. Option1 Set coordinates of vertices

In [ ]:
rectangle_vertices = [
    (-74.02034270713835, 40.69992881162822),  # Southwest corner (longitude, latitude)
    (-74.02034270713835, 40.7111851828668),   # Northwest corner (longitude, latitude)
    (-74.00555129286164, 40.7111851828668),   # Northeast corner (longitude, latitude)
    (-74.00555129286164, 40.69992881162822)   # Southeast corner (longitude, latitude)
]

## 2.2. Option2 Draw a rectangle on a map
Use the panel to set the city, switch drawing modes (Aligned / Rotated / Fixed), and adjust dimensions.

In [ ]:
from voxcity.geoprocessor.draw import create_rectangle_map

ui, rectangle_vertices = create_rectangle_map()
ui

# Step3 Generate voxel city model

In [ ]:
#@title 3.1. Option1 With automatically selected data sources
from voxcity.generator import get_voxcity
meshsize = 5 # meters
voxcity = get_voxcity(rectangle_vertices, meshsize)

### 3.2. Option2 With specified data sources

In [ ]:
#@title 3.2.1 Set data sources and meshsize (m)
building_source = 'OpenStreetMap' #@param ['OpenStreetMap', 'Overture', 'Microsoft Building Footprints', 'Global Building Atlas', 'EUBUCCO v0.1', 'Open Building 2.5D Temporal', 'Local file']
building_complementary_source = "Microsoft Building Footprints" #@param ['None', 'Microsoft Building Footprints', 'Overture', 'OpenStreetMap', 'Global Building Atlas', 'Open Building 2.5D Temporal', 'EUBUCCO v0.1', 'England 1m DSM - DTM', 'Netherlands 0.5m DSM - DTM', 'OpenMapTiles', 'Local file']
land_cover_source = 'OpenStreetMap' #@param ['OpenStreetMap', 'ESA WorldCover', 'OpenEarthMapJapan', 'Urbanwatch', 'ESRI 10m Annual Land Cover', 'Dynamic World V1']
canopy_height_source = 'High Resolution 1m Global Canopy Height Maps v2' #@param ['High Resolution 1m Global Canopy Height Maps v2', 'High Resolution 1m Global Canopy Height Maps', 'ETH Global Sentinel-2 10m Canopy Height (2020)', 'Static']
dem_source = 'USGS 3DEP 10m' #@param ['USGS 3DEP 10m', 'FABDEM', 'COPERNICUS', 'GSI DEM Japan', 'DeltaDTM', 'England 1m DTM', 'DEM France 1m', 'Netherlands 0.5m DTM', 'AUSTRALIA 5M DEM', 'USGS 3DEP 1m', 'NASA', 'Flat']
#@markdown Sources that need no account: OpenStreetMap, Overture, Microsoft Building Footprints, ESA WorldCover, OpenEarthMapJapan, both canopy height maps (v2 recommended), USGS 3DEP 10m, FABDEM, COPERNICUS, GSI DEM Japan. The others are served through Google Earth Engine (Step 1.2).

In [ ]:
#@title 3.2.2 Model generation
from voxcity.generator import get_voxcity
meshsize = 5 # meters
voxcity = get_voxcity(
    rectangle_vertices,
    meshsize,
    building_source=building_source,
    building_complementary_source=building_complementary_source,
    land_cover_source=land_cover_source,
    canopy_height_source=canopy_height_source,
    dem_source=dem_source
)

In [ ]:
#@title (Optional) Load a saved voxel city data
from voxcity.generator.io import load_voxcity
input_path = "output/voxcity.h5"
voxcity = load_voxcity(input_path)

## Step4 Visualize the voxel city model

In [ ]:
#@title 4.1. Option1 Interactive mode
from voxcity.visualizer import visualize_voxcity
visualize_voxcity(voxcity, mode="interactive", voxel_color_map="default", downsample=1)

In [ ]:
#@title 4.1. Option2 Static mode
from voxcity.visualizer import visualize_voxcity
from IPython.display import display, Image

images = visualize_voxcity(voxcity, mode="static", output_directory="output", projection_type="perspective")
for name, path in images:
    display(Image(filename=path))

# Step5 Export files

In [ ]:
#@title 5.1. Export INX file for ENVI-MET
from voxcity.exporter.envimet import export_inx, generate_edb_file

output_directory = './output/envimet'
file_basename = 'voxcity'
export_inx(voxcity, output_directory, file_basename)
generate_edb_file()

In [ ]:
#@title 5.2. VOX file for MagicaVoxel
from voxcity.exporter.magicavoxel import export_magicavoxel_vox

output_path = f"output/magicavoxel"
export_magicavoxel_vox(voxcity, output_path)

In [ ]:
#@title 5.3. OBJ file
from voxcity.exporter.obj import export_obj

output_directory = './output/obj'
output_file_name = 'voxcity'
export_obj(voxcity, output_directory, output_file_name)

# Step6 Urban simulations

# 6.1. Solar radiation

In [ ]:
#@title 6.1.1. Ground-level solar irradiance (instantaneous)
from voxcity.simulator.solar import get_global_solar_irradiance_using_epw

solar_kwargs = {
    "download_nearest_epw": True,  # Whether to automatically download nearest EPW weather file based on location from Climate.OneBuilding.Org
    "calc_time": "01-01 12:00:00",  # Time for instantaneous calculation in format "MM-DD HH:MM:SS"
    "view_point_height": 1.5,  # Height of view point in meters for calculating solar access. Default: 1.5 m
    "output_directory": 'output/test',  # Directory for saving output files
}

# Compute global solar irradiance map (direct + diffuse radiation)
solar_grid = get_global_solar_irradiance_using_epw(
    voxcity,                             # VoxCity object containing voxel data and metadata
    calc_type='instantaneous',           # Calculate instantaneous irradiance at specified time
    **solar_kwargs                       # Pass all the parameters defined above
)

#6.2. View index

In [ ]:
#@title 6.2.1. Green view index, Sky view index
from voxcity.simulator.visibility import get_view_index

view_kwargs = {
    "view_point_height": 1.5,      # Height of observer viewpoint in meters
    "colormap": "viridis",         # Colormap for visualization
    "output_directory": "output",  # Directory to save output files
}

# Compute Green View Index using mode='green'
gvi_grid = get_view_index(voxcity, mode='green', **view_kwargs)

# Adjust parameters for Sky View Index
view_kwargs["colormap"] = "BuPu_r"
view_kwargs["output_file_name"] = "svi"
view_kwargs["elevation_min_degrees"] = 0

# Compute Sky View Index using mode='sky'
svi_grid = get_view_index(voxcity, mode='sky', **view_kwargs)

# Next steps

Building-surface irradiance, landmark visibility and road-network aggregation are in the other tutorials: [demo_solar](demo_solar.ipynb), [demo_view](demo_view.ipynb), [demo_landmark](demo_landmark.ipynb), and the full [tutorial index](index.md).

If VoxCity helps your research, please cite: Fujiwara K. et al. (2026). VoxCity: A Seamless Framework for Open Geospatial Data Integration, Grid-Based Semantic 3D City Model Generation, and Urban Environment Simulation. *Computers, Environment and Urban Systems*, 123, 102366. https://doi.org/10.1016/j.compenvurbsys.2025.102366